In [1]:
!pip install fastapi
!pip install kaleido
!pip install python-multipart
!pip install uvicorn

In [2]:
!git clone --quiet https://github.com/facebookresearch/OrienterNet
%cd /content/OrienterNet
!python -m pip install --progress-bar off --quiet -r requirements/demo.txt
!python -m pip install --quiet -e .
!pip install --upgrade --progress-bar off --quiet plotly

import matplotlib.pyplot as plt
from google.colab import files
def upload_file():
  uploaded = files.upload()
  (path, bin), *_ = uploaded.items()
  with open(path, 'wb') as fid:
    fid.write(bin)
  return path

from maploc.demo import Demo, read_input_image
# Increasing the number of rotations increases the accuracy but requires more GPU memory.
# The highest accuracy is achieved with num_rotations=360
# but num_rotations=64~128 is often sufficient.
# To reduce the memory usage, we can reduce the tile size in the next cell.
demo = Demo(num_rotations=256, device='cpu') # change to "cuda" if you have a GPU.

[WinError 3] 系统找不到指定的路径。: '/content/OrienterNet'
c:\Users\霍昕捷\Documents\WeChat Files\wxid_gmxghjjsb4iv22\FileStorage\File\2026-01


fatal: destination path 'OrienterNet' already exists and is not an empty directory.
d:\Anaconda\envs\orienternet\lib\site-packages\IPython\core\magics\osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements/demo.txt'
ERROR: file:///C:/Users/%E9%9C%8D%E6%98%95%E6%8D%B7/Documents/WeChat%20Files/wxid_gmxghjjsb4iv22/FileStorage/File/2026-01 does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


ModuleNotFoundError: No module named 'google'

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from maploc.demo import Demo, read_input_image
demo = Demo(num_rotations=256, device='cpu')
from maploc.utils.viz_localization import (
    likelihood_overlay,
    plot_dense_rotations,
    add_circle_inset,
)
from maploc.utils.viz_2d import features_to_RGB
import os
from PIL import Image
import torch
import pandas as pd
import csv
folder_path = "/content/drive/MyDrive/images-jpg2"
address = ""
destination_folder_path= '/content/drive/MyDrive/trainimagesCorrect'
tile_size_meters = 128
Original=[]
Corrected=[]
data = []
import numpy as np
# 遍历文件夹中的所有文件
files = os.listdir(folder_path)
i=0
for filename in os.listdir(folder_path):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        # 拼接文件路径
        file_path = os.path.join(folder_path, filename)
    # 构建完整的文件路径
        img= Image.open(file_path)
            # 在这里对图片进行处理
            # 例如，可以调用 image 对象的方法，或者使用 numpy 等库来进行处理
            # 处理完后可以保存图片或者进行其他操作
        print(f"Processing image {i}/{2078}")
        if not address:
            address = None
        from maploc.utils.viz_localization import (
            likelihood_overlay,
            plot_dense_rotations,
            add_circle_inset,
            )
        image, camera, gravity, proj, bbox, prior_latlon = read_input_image(
                file_path,
                prior_address=address,
                tile_size_meters=tile_size_meters,
            )
        from maploc.osm.tiling import TileManager
        tiler = TileManager.from_bbox(proj,bbox + 10, demo.config.data.pixel_per_meter)
        canvas = tiler.query(bbox)
# Run the inference
        uv, yaw, prob, neural_map, image_rectified = demo.localize(
          image, camera,canvas,roll_pitch=gravity)
        Original.append(prior_latlon[:2])
        Corrected.append(proj.unproject(canvas.to_xy(uv)))
        s = 1/25
        thresh = 0.1
        k = 3
        t = torch.argmax(prob, -1)

        yaws = t.numpy() / prob.shape[-1] * 360
        prob_0 = prob.max(-1).values / prob.max()
        mask = prob_0 > thresh
        masked = prob_0.masked_fill(~mask, 0)
        max_ = torch.nn.functional.max_pool2d(
            masked.float()[None, None], k, stride=1, padding=k // 2
            )
        mask=(max_[0, 0] == masked.float()) & mask
        indices = np.where(mask.numpy() > 0)
        temp = indices[::-1]
        u, v = uv[0], uv[1]
        x1 = float(Corrected[i][0])
        y1 = float(Corrected[i][1])
        for j in range(len(temp[0])):
            if u == temp[0][j]:
                if v == temp[1][j]:
                    temp_0 = yaws[indices]
                    angle=float(temp_0[j])
                    new_filename = f'{x1}_{y1}_{angle}.jpg'  # 新的文件名格式，可以根据需要修改
                    data.append({'Filename': filename, 'New_Filename': new_filename})
                    new_path = os.path.join(destination_folder_path, new_filename)
                    img.save(new_path)
                    i+=1
                    csv_file_path = '/content/drive/MyDrive/gtCorrect-train_result.csv'
with open(csv_file_path, 'w', newline='') as csvfile:
    fieldnames = ['Filename', 'New_Filename']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for item in data:
        writer.writerow(item)

print("CSV 文件已生成:", csv_file_path)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded as API: https://jinlinyi-perspectivefields.hf.space/ ✔


[2024-04-11 12:13:21 maploc INFO] Using prior location from EXIF.
[2024-04-11 12:13:21 maploc INFO] Calling the PerspectiveFields calibrator, this may take some time.


Processing image 0/2078


[2024-04-11 12:13:44 maploc INFO] Using (roll, pitch) (-5.51, 4.71).
[2024-04-11 12:13:44 maploc INFO] Calling the OpenStreetMap API...
[2024-04-11 12:14:56 maploc INFO] Using prior location from EXIF.
[2024-04-11 12:14:56 maploc INFO] Calling the PerspectiveFields calibrator, this may take some time.


Processing image 1/2078


[2024-04-11 12:15:06 maploc INFO] Using (roll, pitch) (3.86, -0.23).
[2024-04-11 12:15:06 maploc INFO] Calling the OpenStreetMap API...


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
